# Capstone — Modern Data Engineering for AI Systems

**Team:** Alanoud Alotaibi, Rawan H., Reem Alshathri | **Program:** SDAIA Academy | **Trainer:** Mohammed Albeladi  
**Dataset:** [Customer Support Tickets - CRM Dataset](https://www.kaggle.com/datasets/ajverse/customer-support-tickets-crm-dataset)

---

## 📋 Executive Overview & Core Deliverables

This production notebook executes a 5-part data engineering and AI pipeline:

1. **Deliverable 1: Streaming Ingestion & Quality Contract** — Real `kafka-python` Producer/Consumer, Pydantic schema validation contract, Dead Letter Queue (DLQ) routing, and quarantine logging.
2. **Deliverable 2: Delta Lakehouse Architecture** — Bronze/Silver/Gold layers using PySpark and Delta Lake, real `DeltaTable.merge()` upsert on `Ticket_ID`, schema enforcement verification, and business aggregations.
3. **Deliverable 3: Production Hybrid RAG Pipeline** — Text chunking with overlap, ChromaDB vector store, `BM25Okapi` keyword search, Reciprocal Rank Fusion (RRF), Cross-Encoder reranking, grounded answers, and exact ticket citations.
4. **Deliverable 4: Pipeline Orchestration** — Apache Airflow DAG (`sdaia_capstone_pipeline_dag`) built with TaskFlow API (`@dag`, `@task`), explicit task dependencies, and downstream quality gate halting.
5. **Deliverable 5: Data Quality Gate & Lineage Observability** — Great Expectations quality suite enforcing an 80% threshold, and OpenLineage lifecycle event tracking emitting `START`, `COMPLETE`, and `FAIL` JSON payloads.

---

## Step 0 — Repository & Environment Setup

In [ ]:
import sys, os

# Add project root to sys.path
sys.path.insert(0, os.getcwd())

from src.config import IN_COLAB, IN_WINDOWS, RAW_DIR, DELTA_DIR, CHROMA_DIR
print(f"✓ Environment Setup: IN_COLAB={IN_COLAB}, IN_WINDOWS={IN_WINDOWS}")
print(f"✓ Project Root: {os.getcwd()}")

## Step 1 — Install Production Libraries

In [ ]:
!pip install -q pyspark==3.5.0 delta-spark==3.2.0 kafka-python==2.0.2 "pydantic>=2.0" \
    kagglehub chromadb sentence-transformers rank-bm25 "apache-airflow>=2.8" \
    "great-expectations>=1.0" openlineage-python

## Step 2 — Execute Pipeline End-to-End

In [ ]:
from src.dag_pipeline import run_pipeline

# Run the full end-to-end pipeline on the CRM dataset
results = run_pipeline("customer_support_tickets.csv")

## Step 3 — Verify Deliverable 1: Ingestion & DLQ Quarantine

In [ ]:
from src.kafka_io import consume_and_validate
from src.synthetic_data import generate_synthetic_tickets

# Test DLQ with malformed records injection (10% bad rows)
test_records = generate_synthetic_tickets(50, bad_row_fraction=0.10).to_dict("records")
val_out = consume_and_validate(records_input=test_records)

print(f"✓ Processed: {val_out['total_processed']} | Valid: {val_out['valid_count']} | Quarantined to DLQ: {val_out['invalid_count']}")
if val_out['invalid_records']:
    print("Sample DLQ Rejection:", val_out['invalid_records'][0]['rejection_reason'])

## Step 4 — Verify Deliverable 2: Delta Lakehouse & Gold Metrics

In [ ]:
from src.lakehouse import get_spark, build_gold, verify_schema_enforcement

spark = get_spark()
gold_tables = build_gold(spark)
print("✓ Schema Enforcement Verified:", verify_schema_enforcement(spark))
print("✓ Gold Category Metrics built successfully.")

## Step 5 — Verify Deliverable 3: Hybrid RAG & Grounded Citations

In [ ]:
from src.tasks import task_rag

rag_out = task_rag("Data not syncing across devices")
print("✓ RAG Response:\n", rag_out["answer"])

## Step 6 — Verify Deliverables 4 & 5: Quality Gate & OpenLineage

In [ ]:
from src.quality import run_quality_gate
from src.tasks import task_quality_gate

q_res = task_quality_gate()
print(f"✓ Quality Gate Audit Status: {q_res['status']} | Quality Score: {q_res['quality_score']:.2%}")